# 🏦 SME Finance Copilot — QLoRA Cloud Training (Free T4 GPU)

This notebook fine-tunes **Qwen-2.5-3B-Instruct** on **4,850 verified financial examples**.

### Why Train on Colab?
- **Zero Laptop Heat / Power**: Saves laptop hardware from thermal shutdowns.
- **Free 16GB T4 GPU**: 3x more VRAM than laptop GPUs.
- **Turnkey Download**: Automatically packages and downloads `sme-finance-lora.zip` (~85 MB) to your local machine when complete.

---

### Step 1: Verify GPU Allocation

In [ ]:
!nvidia-smi

### Step 2: Install Fine-Tuning Dependencies

In [ ]:
!pip install -q torch transformers peft bitsandbytes datasets trl accelerate

### Step 3: Upload the Dataset (`colab_dataset.zip`)
Run this cell and select `colab_dataset.zip` from your local `dataset/` directory (size: ~2.2 MB).

In [ ]:
from google.colab import files
import os

print("Please select 'colab_dataset.zip' (from your project dataset/ folder):")
uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith('.zip'):
        !unzip -o {fn}
        print(f"Extracted {fn} successfully!")

### Step 4: Run QLoRA Training (~12-15 Minutes)
Trains on 4,850 examples across 9 financial categories with 4-bit NF4 quantization and FP16 mixed precision.

In [ ]:
import os
import signal
import shutil
import gc
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = torch.cuda.mem_get_info()[0] / (1024**3)
    if free_gb < 10.0:
        print(f"[AUTO-CLEAN] Dirty GPU memory detected ({free_gb:.2f} GB free). Purging dead process now...")
        os.kill(os.getpid(), signal.SIGKILL)

from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from google.colab import files

model_id = "Qwen/Qwen2.5-3B-Instruct"
output_dir = "sme-finance-lora"

# 1. 4-Bit NF4 Quantization with FP16 compute for Tesla T4 GPU
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# 2. Pin model directly to GPU 0
print("[1/5] Loading base model in float16...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.config.use_cache = False

# 3. Configure LoRA on all projections
print("[2/5] Configuring LoRA adapters...")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 4. Load ChatML Dataset
print("[3/5] Loading ChatML dataset...")
train_dataset = load_dataset("json", data_files="finetune_train.jsonl", split="train")

def format_chatml(batch):
    formatted = []
    for conversation in batch["messages"]:
        text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
        formatted.append(text)
    return {"text": formatted}

train_dataset = train_dataset.map(format_chatml, batched=True, load_from_cache_file=False)

# 5. SFTTrainer Execution with use_reentrant=False (~273 steps, ~35 minutes, ~3.8 GB VRAM)
print("[4/5] Starting fine-tuning loop...")
sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    dataloader_pin_memory=True,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
    max_length=1024,
    dataset_text_field="text",
    eval_strategy="no",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=sft_config,
)

trainer.train()

# 6. Save Adapter & Download
print("[5/5] Saving adapter and preparing download...")
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

shutil.make_archive("sme-finance-lora", "zip", output_dir)
print("Download started for sme-finance-lora.zip (~85 MB)!")
files.download("sme-finance-lora.zip")


### Step 5: After Download Finishes
1. Extract `sme-finance-lora.zip` into `f:\SME Finance Copilot\models\sme-finance-lora\`
2. On your laptop, run:
   ```powershell
   .\.venv-train\Scripts\python.exe scripts/verify_local_inference.py
   ```
3. Inference runs in 4-bit at only 2.2 GB VRAM with near-zero power draw (~15W).